Auto Loader is Databricks
Auto Loader is Databricks’ cloud-native file ingestion engine for ingesting new files incrementally from object storage.

#####Supported Sources:

- AWS S3
- Azure ADLS Gen2
- Google Cloud Storage (GCS)

#####Modes:

- Directory listing - Directory listing scans storage paths to detect new files (This works in free edition)
- File notification - Processes files as soon as they arrive at scale (This will not work in free edition because the cloud storage event trigger can't control/trigger Databricks LF Ingestion)

#####Directory listing (Databricks Lakeflow Ingestion - Autoloader - Directory Listing)

- Spark lists the Cloud directory (pull model)
- Detects new files (Incremental Autoloader)
- Infers schema / evolves if needed
- Copy the file(s) & store the schema info in a schema file, so further schema inference is not needed.
- After file1 is copied to Bronze layer -> Updates checkpoint (maintaining the file info of whichever is copied already)
- Waits for next trigger of the Lakeflow pipeline and follow step 1 to 5.

#####File Notification (we will see it in the cloud databricks version)

- Cloud storage emits file-create event (S3 Event, ADLS Event Grid, GCS Pub/Sub)
- Event is delivered to Databricks queue
- Auto Loader receives notification (push model)
- New file is registered
- Infers schema / evolves if needed
- Copy the file(s) & store the schema info in a schema file, so further schema inference is not needed.
- Updates checkpoint (file1 is processed...)
- Stream stays idle until next event arrives

#####Benifits of Autoloader:

- Incremental and Efficient File Ingestion: Auto Loader automatically detects and processes new files as they arrive in your source directory (e.g., S3 or Unity Catalog volume). This eliminates manual tracking and reprocessing, ensuring only new data is ingested each run.
- Schema Evolution Support: With options like "cloudFiles.schemaEvolutionMode": "addNewColumns" and "mergeSchema": "true", Auto Loader can handle changes in your data schema over time, adding new columns without breaking your pipeline.
- Scalability and Resource Optimization: Properties such as "cloudFiles.maxFilesPerTrigger" allow you to control how many files are processed per batch, helping manage resource usage and scale to large datasets.
- Checkpointing and Fault Tolerance: Auto Loader maintains checkpoints and schema locations, so it can resume from where it left off in case of failures, ensuring reliable and consistent data ingestion.
- Unified Streaming and Batch Processing: By using readStream and writeStream, your pipeline can handle both streaming and batch workloads seamlessly, making it suitable for real-time and scheduled data ingestion.

#####To perform schema evolution, we have to use the below properties:
######Read side:
.option("cloudFiles.schemaEvolutionMode","addNewColumns")

######Write side:
.option("mergeSchema", "true")

In [0]:
# Define the cloud source path (this can be S3, ADLS, or GCS)
cloudsrc = "/Volumes/catalog1_we47/schema1_we47/clouddatalake/sourcesystemdata/"
# Example alternative for GCS:
# cloudsrc = "gs://izsourcebucket/Master_City_List_hour1.csv"

# Define the bronze target path where ingested data will be stored
bronzetgt = "/Volumes/catalog3_we47/schema3_we47/datalake/bronze/ourtargetlocation/"

# Define checkpoint location – Spark Structured Streaming uses this to track progress
# It stores metadata about which files have already been processed
ckptlocation = "/Volumes/catalog1_we47/schema1_we47/clouddatalake/ckpt/_checkpoint"

# Define schema location – stores the inferred schema of the source data
# Useful for schema evolution when new columns are added
schemalocation = "/Volumes/catalog1_we47/schema1_we47/clouddatalake/_schema"

# Read streaming data using Auto Loader (cloudFiles)
df1 = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.maxFilesPerTrigger", 1) \
    .option("cloudFiles.inferColumnTypes", True) \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .option("checkpointLocation", ckptlocation) \
    .option("cloudFiles.schemaLocation", schemalocation) \
    .option("header", True) \
    .load(cloudsrc)                                      # Load from cloud source (S3/ADLS/GCS)

# Note:
# - .option("cloudFiles.useNotifications", "true") can be used for event-based file discovery.
#   If removed, Auto Loader falls back to directory listing.
# - maxFilesPerTrigger helps throttle ingestion rate. Even if set to 1, all files will eventually be processed.

In [0]:
# Write the streaming DataFrame 'df1' into the bronze target location
df1.writeStream \
    .trigger(availableNow=True) \
    .option("checkpointLocation", ckptlocation) \
    .option("cloudFiles.schemaLocation", schemalocation) \
    .option("mergeSchema", "true") \
    .start(bronzetgt)                             # Start the streaming write into the bronze target path

In [0]:
# Read the Delta table stored at the bronze target location
df_bronze = spark.read.format("delta").load(bronzetgt)

# Order the records by the column 'city_name'
# Display the first 100 rows for inspection
df_bronze.orderBy("city_name").show(100)